In [1]:
# モジュールのインポート
import os
import time
import numpy as np
import pandas as pd
import optuna
import random
from optuna.pruners import MedianPruner
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score

In [ ]:
# ===== 再現性確保のためのシード固定 =====

SEED = 42

def set_seed(seed: int = SEED):
    """random / numpy / torch (CPU・GPU) のシードを固定し、cuDNN を決定論的にする。"""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)


In [3]:
dataset = pd.read_csv("dataset/くま発見3次メッシュデータセット_東北全県.csv")
dataset["kuma_view"] = 1*(dataset["kuma_view"]>0)

In [4]:
import pickle
def seve_metrics(name, datalist):
    with open('optina_pickeles/' + str(name) + '_train_loss_list.pickle', mode='wb') as fo:
        pickle.dump(datalist[0], fo)
    with open('optina_pickeles/' + str(name) + '_val_loss_list.pickle', mode='wb') as fo:
        pickle.dump(datalist[1], fo)
    with open('optina_pickeles/' + str(name) + '_test_loss_list.pickle', mode='wb') as fo:
        pickle.dump(datalist[2], fo)
        
    with open('optina_pickeles/' + str(name) + '_train_acc_list.pickle', mode='wb') as fo:
        pickle.dump(datalist[3], fo)
    with open('optina_pickeles/' + str(name) + '_val_acc_list.pickle', mode='wb') as fo:
        pickle.dump(datalist[4], fo)
    with open('optina_pickeles/' + str(name) + '_test_acc_list.pickle', mode='wb') as fo:
        pickle.dump(datalist[5], fo)
 
    with open('optina_pickeles/' + str(name) + '_train_positive_list.pickle', mode='wb') as fo:
        pickle.dump(datalist[6], fo)
    with open('optina_pickeles/' + str(name) + '_val_positive_list.pickle', mode='wb') as fo:
        pickle.dump(datalist[7], fo)
    with open('optina_pickeles/' + str(name) + '_test_positive_list.pickle', mode='wb') as fo:
        pickle.dump(datalist[8], fo)
        
    with open('optina_pickeles/' + str(name) + '_train_F1_list.pickle', mode='wb') as fo:
        pickle.dump(datalist[9], fo)        
    with open('optina_pickeles/' + str(name) + '_val_F1_list.pickle', mode='wb') as fo:
        pickle.dump(datalist[10], fo)
    with open('optina_pickeles/' + str(name) + '_test_F1_list.pickle', mode='wb') as fo:
        pickle.dump(datalist[11], fo)
        
    with open('optina_pickeles/' + str(name) + '_train_AUC_list.pickle', mode='wb') as fo:
        pickle.dump(datalist[12], fo)        
    with open('optina_pickeles/' + str(name) + '_val_AUC_list.pickle', mode='wb') as fo:
        pickle.dump(datalist[13], fo)
    with open('optina_pickeles/' + str(name) + '_test_AUC_list.pickle', mode='wb') as fo:
        pickle.dump(datalist[14], fo)

    with open('optina_pickeles/' + str(name) + '_train_meanscore_list.pickle', mode='wb') as fo:
        pickle.dump(datalist[15], fo)
    with open('optina_pickeles/' + str(name) + '_valid_meanscore_list.pickle', mode='wb') as fo:
        pickle.dump(datalist[16], fo)
    with open('optina_pickeles/' + str(name) + '_test_meanscore_list.pickle', mode='wb') as fo:
        pickle.dump(datalist[17], fo)
    if len(datalist) > 20:
        with open('optina_pickeles/' + str(name) + '_train_PRAUC_list.pickle', mode='wb') as fo:
            pickle.dump(datalist[18], fo)
        with open('optina_pickeles/' + str(name) + '_val_PRAUC_list.pickle', mode='wb') as fo:
            pickle.dump(datalist[19], fo)
        with open('optina_pickeles/' + str(name) + '_test_PRAUC_list.pickle', mode='wb') as fo:
            pickle.dump(datalist[20], fo)

In [5]:
from torch.utils.data import TensorDataset
Tensor_dataset = torch.load("dataset/dataset_tensor_CNN.pt", map_location="cpu")

/tmp/ipykernel_3722254/1928488959.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  Tensor_dataset = torch.load("../../dataset_tensor_CNN.pt", map_location="cpu")


In [ ]:
# 利用可能なGPU一覧
n_gpus = torch.cuda.device_count()
print("GPU count:", n_gpus)
for i in range(n_gpus):
    print(i, torch.cuda.get_device_name(i))

# ここでは 6 枚全部を使う前提
DEVICE_IDS = list(range(n_gpus))[:2]  # [0,1,2,3,4,5]

# DataParallelの"メイン"になるGPU
MAIN_DEVICE_ID = DEVICE_IDS[0]
device = torch.device(f"cuda:{MAIN_DEVICE_ID}")
torch.cuda.set_device(MAIN_DEVICE_ID)

GPU count: 6
0 NVIDIA RTX A5000
1 NVIDIA RTX A5000
2 NVIDIA RTX A5000
3 NVIDIA RTX A5000
4 NVIDIA RTX A5000
5 NVIDIA RTX A5000


In [7]:
from torch.utils.data.dataset import Subset

In [8]:
import torch
import torch.nn as nn

class ResBlock(nn.Module):
    """stride 付き畳み込みで空間ダウンサンプリングする BasicBlock"""
    def __init__(self, in_ch: int, out_ch: int, stride: int = 1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, kernel_size=4, stride=stride, padding=2, bias=False)
        self.bn1   = nn.BatchNorm2d(out_ch)
        self.relu  = nn.ReLU(inplace=True)
        self.pool  = nn.MaxPool2d(4)
        # self.conv2 = nn.Conv2d(out_ch, out_ch, kernel_size=3, stride=1, padding=1, bias=False)
        # self.bn2   = nn.BatchNorm2d(out_ch)
        
        self.downsample = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=1, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.MaxPool2d(4)
        )
        #self.downsample = nn.Conv2d(in_ch, out_ch, kernel_size=1, stride=stride, padding=0, bias=False)

    def forward(self, x):
        identity = x
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        out = self.pool(x)
        #print(out.shape)
        #print(self.downsample(identity).shape)
        # out = self.bn2(self.conv2(out))
        
        out = self.relu(out + self.downsample(identity))
        return out


class CNN(nn.Module):
    def __init__(self, base_channels: int = 16, dropout: float = 0.0, m1: int = 2048, m2: int = 1024):
        super(CNN, self).__init__()

        # --- x(3ch) ブランチ ---

        self.layer0 = nn.Sequential(
            nn.Conv2d(3, base_channels, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(base_channels),
            nn.ReLU(inplace=True),
        )
        self.layer0 = ResBlock(24, base_channels, stride=1)
        #self.layer1 = ResBlock(3, base_channels, stride=1)
        #self.layer1B = ResBlock(10, base_channels, stride=1)
        #self.layer1C = ResBlock(12, base_channels, stride=1)
        self.layer2 = ResBlock(base_channels, base_channels, stride=1)
        self.layerA = nn.Sequential(
            nn.Conv2d(in_channels = 5, out_channels = 5, kernel_size=4),
            nn.BatchNorm2d(5),
            nn.ReLU(),
            nn.MaxPool2d(4))  
        self.fc  = nn.Sequential(nn.LazyLinear(m1),nn.ReLU())
        # self.fc  = nn.Sequential(nn.Linear(m1, m1),nn.ReLU())
        self.fcA = nn.Sequential(nn.Linear(m1, m2), nn.ReLU())
        self.fc2 = nn.LazyLinear(1)
        self.dropout = nn.Dropout(dropout)
        self.sigmoid = nn.Sigmoid()  # BCEWithLogitsLoss を使う場合は削除

    def forward(self, x, y, z, B, C):
        #x = self.layer0(x)
        """
        x = self.layer1(x)
        x = self.layer2(x)
        x = x.view(x.size(0), -1)
        B = self.layer1B(B)
        B = self.layer2(B)
        B = B.view(B.size(0), -1)
        C = self.layer1C(C)
        C = self.layer2(C)
        C = C.view(C.size(0), -1)
        """
        # print(x.shape, B.shape, C.shape)
        x = torch.flatten(x, start_dim=1)
        y = torch.flatten(y, start_dim=1)
        z = torch.flatten(z, start_dim=1)
        B = torch.flatten(B, start_dim=1)
        C = torch.flatten(C, start_dim=1)
        x = torch.cat([x, B, C, y, z], dim=1)   # [batch, 特徴量] のまま渡す
        x = self.fc2(x)                          # torch.flatten(x) の行を削除
        x = self.sigmoid(x)
        return x

def train(model, train_loader, criterion, device,optimizer):
    model.train()
    running_loss = 0
    correct = 0
    total = 0; positive = 0; sumout = 0; sumlabel = 0
    for images,images2,images3, images4B, images4C, labels in train_loader:
    # for images,images2,images3,labels in train_loader:
        images = images.float().to(device, non_blocking=True)
        images2 = images2.float().to(device, non_blocking=True)
        images3 = images3.float().to(device, non_blocking=True)
        images4B = images4B.float().to(device, non_blocking=True)
        images4C = images4C.float().to(device, non_blocking=True)
        labels = labels.float().to(device, non_blocking=True)
        optimizer.zero_grad()
        outputs = model(images, images2, images3, images4B, images4C)
        # outputs = model(images, images2, images3)
        loss = criterion(outputs.squeeze(1).float(), labels.float())
        running_loss += loss
        loss.backward()
        optimizer.step()
        correct += sum((outputs.squeeze(1) > 0.5) == (labels> 0.5)) / len(labels)
        total += labels.size(0)
        sumlabel += sum((labels > 0.5))
        positive += sum((outputs > 0.5))
        sumout += sum(outputs)
    train_loss = running_loss
    train_acc = correct / len(train_loader) 
    #print("Percent of Positive in sample:: ", sumlabel / total, positive / total, sumout / total)
    return None

import time
import torch
import torch.nn as nn


# 追加: 必要な import
from sklearn.metrics import f1_score, average_precision_score

def _forward_batch(model, batch, device, logits_out=False):
    """
    TensorDataset の最後の要素を y、それ以外を X* として取り出し forward するヘルパ。
    - 複数ブランチ入力に対応: model(*Xs) を試み、失敗すれば model(Xs[0]) にフォールバック。
    - BCEと出力の整合:
        - BCEWithLogitsLoss を使う場合: logits_out=True（Sigmoidしない）
        - BCELoss を使う場合: logits_out=False（Sigmoid済み出力 or 手動でSigmoid）
    """
    *Xs, y = batch
    Xs = [x.float().to(device, dtype=torch.float32, non_blocking=True) for x in Xs]
    y = y.float().to(device, dtype=torch.float32, non_blocking=True)
    
    try:
        out = model(*Xs)
    except TypeError:
        out = model(Xs[0])
    if out.ndim > 1 and out.size(-1) == 1:
        out = out.squeeze(-1)

    if logits_out:               # BCEWithLogitsLoss のとき
        logits = out
        probs = torch.sigmoid(out)
    else:                        # BCELoss のとき（出力がSigmoid済み前提）
        logits = out
        probs = out

    return logits, probs, y

from sklearn.metrics import roc_auc_score

@torch.no_grad()
def eval_epoch(model, loader, criterion, device, logits_out=False, use_streaming_auc=True):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    y_true_all, y_prob_all = [], []
    y_pred_all = []
    pos_mean_list, score_mean_list = [], []

    for batch in loader:
        logits, probs, y = _forward_batch(model, batch, device, logits_out=logits_out)
        loss = criterion(probs if isinstance(criterion, torch.nn.BCELoss) else logits, y)

        running_loss += loss.item() * y.size(0)
        preds = (probs >= 0.5).long()
        correct += (preds == y.long()).sum().item()
        total   += y.size(0)

        # AUC用に確率と正解を蓄積（2値前提：陽性確率は probs そのもの）
        y_true_all.append(y.detach().long().cpu().numpy())
        y_prob_all.append(probs.detach().float().cpu().numpy())
        y_pred_all.append(preds.detach().long().cpu().numpy())
        pos_mean_list.append(y.float().mean().item())
        score_mean_list.append(probs.float().mean().item())

    valid_loss = running_loss / max(total, 1)
    valid_acc  = correct / max(total, 1)

    y_true_all = np.concatenate(y_true_all) if y_true_all else np.array([])
    y_prob_all = np.concatenate(y_prob_all) if y_prob_all else np.array([])
    y_pred_all = np.concatenate(y_pred_all) if y_pred_all else np.array([])

    # --- AUCの算出 ---
    auc_val = float("nan")
    if y_true_all.size > 0:
        try:
            if use_streaming_auc and 'auc_binary_streaming' in globals():
                # 既存のストリーミングAUC関数を優先
                auc_val = auc_binary_streaming(model, loader, device=device, thresholds=1028)
                auc_val = float(getattr(auc_val, "item", lambda: auc_val)())
            else:
                # フォールバック：sklearn
                # 片側しか出現しないfoldでは例外が出るためtry/except
                auc_val = roc_auc_score(y_true_all, y_prob_all)
        except Exception:
            auc_val = float("nan")

    # 参考：F1など他指標を継続して返す場合
    from sklearn.metrics import f1_score, average_precision_score
    valid_f1 = f1_score(y_true_all, y_pred_all) if y_true_all.size > 0 else 0.0

    valid_positive  = float(np.mean(pos_mean_list)) if pos_mean_list else 0.0
    valid_meanscore = float(np.mean(score_mean_list)) if score_mean_list else 0.0

    try:
        ap_val = average_precision_score(y_true_all, y_prob_all) if y_true_all.size > 0 else float("nan")
    except Exception:
        ap_val = float("nan")
    return valid_loss, valid_acc, valid_positive, valid_f1, valid_meanscore, auc_val, ap_val


In [9]:
# 5. 空間的クロスバリデーションによる学習と評価 + Optuna
Lcluster = []; LACC = []; Lauc = []
try:
    final_df = pd.read_csv("optuna_results/宮城県-lr-optina-植生あり.csv")
except:
    final_df = pd.DataFrame(columns=["pref", "n", "epoch","learning_rate","weight_decay","batch_size", "m1_size","m2_size", "dropout","base_channels",
                    "train_y", "train_acc", "train_positive", "train_meanscore","valid_y","valid_acc","valid_auc","CNN_valid_auc", "valid_positive", "valid_meanscore",
                    "test_y", "test_acc", "test_auc","CNN_test_auc","test_positive",  "test_meanscore"
                ])
    final_df.to_csv("optuna_results/宮城県-lr-optina-植生あり.csv")
num_n = len(final_df)
Nth = 1
patience = 10 # アーリーストッピング用
FOLDS = [("宮城県","山形県"),("山形県","岩手県"),("岩手県","福島県"),("福島県","秋田県"),("秋田県","宮城県")]
for n, (TEST_PREF, VALID_PREF) in enumerate(FOLDS):
    CSV_PATH = f"optuna_results/{TEST_PREF}-lr-optina-植生あり.csv"
    if not os.path.exists(CSV_PATH):
        pd.DataFrame(columns=final_df.columns).to_csv(CSV_PATH)

    print(f"Training on {n} th set")

    num_epochs = 200
    patience = 10 # アーリーストッピング用

    WAIT_SECONDS = 1800          # 例：3分待つ
    MAX_RETRIES_PER_TRIAL = 5   # 例：1 trial につき 3回までリトライ

    # ---- 目的関数：このfoldのsplitを固定したまま、trialごとに学習 ----
    _GLOBAL_BEST_AUC = [-1.0]
    def objective(trial: optuna.Trial):
        for attempt in range(1, MAX_RETRIES_PER_TRIAL + 1):
            try:
                print(f"[Trial {trial.number}] Attempt {attempt}/{MAX_RETRIES_PER_TRIAL}")

                ########## データ分割: test=宮城県, valid=山形県, train=その他の県 ##########
                # valid/train は県単位の固定セット（shuffle は並び替えのみで構成は不変）→ trial 間で split 固定
                test_index  = list(dataset.index[dataset["prefecture"] == TEST_PREF])
                valid_index = list(dataset.index[dataset["prefecture"] == VALID_PREF])
                train_index = list(dataset.index[(dataset["prefecture"] != TEST_PREF) & (dataset["prefecture"] != VALID_PREF)])
                random.seed(SEED)  # 分割・ダウンサンプリングを再現可能に
                random.shuffle(test_index)
                random.shuffle(valid_index)
                random.shuffle(train_index)

                # train の負例(kuma_view==0)を 1/10 にダウンサンプリング（正例は全て保持）
                _labels = dataset["kuma_view"]
                train_pos = [i for i in train_index if _labels[i] == 1]
                train_neg = [i for i in train_index if _labels[i] == 0]
                random.shuffle(train_neg)
                train_neg = train_neg[: len(train_neg) // 10]
                train_index = train_pos + train_neg
                random.shuffle(train_index)
                                
                train_idx = torch.as_tensor(train_index, dtype=torch.long)
                valid_idx = torch.as_tensor(valid_index, dtype=torch.long)
                test_idx  = torch.as_tensor(test_index,  dtype=torch.long)
                
                train_dataset = Subset(Tensor_dataset, train_idx.tolist())
                valid_dataset = Subset(Tensor_dataset, valid_idx.tolist())
                test_dataset  = Subset(Tensor_dataset, test_idx.tolist())
                
                '''誤差(loss)を記録する空の配列を用意'''
                loss_list = []
                train_loss_list = []; train_acc_list = [];train_positive_list = []; train_F1_list = [];train_meanscore_list = [];
                valid_loss_list = []; valid_acc_list = [];valid_positive_list = []; valid_F1_list = []; valid_auc_list = [];valid_meanscore_list = []
                test_loss_list = []; test_acc_list = [];test_positive_list = []; test_F1_list = []; test_auc_list = [];test_meanscore_list = []
                train_auc_list = []
                train_prauc_list = []; valid_prauc_list = []; test_prauc_list = []
                # train_meanscore_list と train_F1_list はすでに上で初期化済み

                # 探索パラメータ
                lr = trial.suggest_float("lr", 1e-4, 3e-3, log=True)
                weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True)
                batch_size = trial.suggest_categorical("batch_size", [32, 64, 128])
                dropout = trial.suggest_float("dropout", 0.0, 0.3)
                base_ch = trial.suggest_categorical("base_channels", [16,32])
                m1_size = trial.suggest_categorical("m1_size", [512, 1024, 2048, 4196])
                m2_size = trial.suggest_categorical("m2_size", [256, 512, 1024])

                # DataLoader（trialのbatch_size）
                train_loader_for_train = DataLoader(
                    train_dataset,
                    batch_size=batch_size*6,
                    shuffle=True,
                    pin_memory=True,
                    num_workers=24,
                    prefetch_factor=4,
                    persistent_workers=False
                )
                train_loader = DataLoader(
                    train_dataset,
                    batch_size=256,
                    shuffle=False,
                    pin_memory=True,
                    num_workers=24,
                    prefetch_factor=4,
                    persistent_workers=False
                )
                valid_loader = DataLoader(
                    valid_dataset,
                    batch_size=256,
                    shuffle=False,
                    pin_memory=True,
                    num_workers=24,
                    prefetch_factor=4,
                    persistent_workers=False
                )
                test_loader  = DataLoader(
                    test_dataset,
                    batch_size=256,
                    shuffle=False,
                    pin_memory=True,
                    num_workers=24,
                    prefetch_factor=4,
                    persistent_workers=False
                )

                # モデル・最適化
                model = CNN(base_channels = base_ch, dropout = dropout, m1 = m1_size, m2 = m2_size).float()
                model.to(device)
                criterion = torch.nn.BCELoss()
                optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

                # ログ蓄積（必要に応じて保持）
                best_val_auc = -1.0
                best_val_acc = 0.0
                best_test_auc = 0.0
                best_test_acc = 0.0
                best_epoch = -1
                best_val_prauc = float("nan"); best_test_prauc = float("nan"); best_train_prauc = float("nan")
                wait = 0

                for epoch in range(num_epochs):
                    # トレーニング
                    model.train()
                    train(model, train_loader_for_train, criterion, device, optimizer)
                
                    # 評価
                    model.eval()
                    train_loss, train_acc, train_positive, train_F1, train_meanscore, train_auc, train_prauc = \
                        eval_epoch(model, train_loader, criterion, device, logits_out=False, use_streaming_auc=True)
                    valid_loss, valid_acc, valid_positive, valid_F1, valid_meanscore, valid_auc, valid_prauc = \
                        eval_epoch(model, valid_loader, criterion, device, logits_out=False, use_streaming_auc=True)
                    test_loss, test_acc, test_positive, test_F1, test_meanscore, test_auc, test_prauc = \
                        eval_epoch(model, test_loader,  criterion, device, logits_out=False, use_streaming_auc=True)

                    train_loss_list.append(train_loss); train_acc_list.append(train_acc);train_positive_list.append(train_positive); train_F1_list.append(train_F1)
                    valid_loss_list.append(valid_loss); valid_acc_list.append(valid_acc);valid_positive_list.append(valid_positive); valid_F1_list.append(valid_F1)
                    test_loss_list.append(test_loss); test_acc_list.append(test_acc);test_positive_list.append(test_positive); test_F1_list.append(test_F1)
                    train_meanscore_list.append(train_meanscore);valid_meanscore_list.append(valid_meanscore);test_meanscore_list.append(test_meanscore)
                    train_auc_list.append(train_auc); valid_auc_list.append(valid_auc); test_auc_list.append(test_auc); train_prauc_list.append(train_prauc); valid_prauc_list.append(valid_prauc); test_prauc_list.append(test_prauc)

                    if valid_auc > best_val_auc:
                        best_val_auc = valid_auc
                        best_val_acc = getattr(valid_acc, "item", lambda: valid_acc)()
                        best_test_auc = test_auc
                        best_test_acc = getattr(test_acc, "item", lambda: test_acc)()
                        best_epoch = epoch + 1
                        best_val_prauc = valid_prauc; best_test_prauc = test_prauc; best_train_prauc = train_prauc
                        # --- per-trial best-epoch モデル保存（state_dict＋メタ情報）---
                        _m = model.module if isinstance(model, torch.nn.DataParallel) else model
                        _pkg = {"model": "lr", "trial": int(trial.number), "epoch": int(best_epoch),
                                "valid_auc": float(best_val_auc), "params": dict(trial.params),
                                "state_dict": {k: v.detach().cpu().clone() for k, v in _m.state_dict().items() if not isinstance(v, (torch.nn.parameter.UninitializedParameter, torch.nn.parameter.UninitializedBuffer))}}
                        os.makedirs("optuna_models", exist_ok=True)
                        torch.save(_pkg, f"optuna_models/lr_{TEST_PREF}_trial{int(trial.number)}_best.pkl")
                        if float(best_val_auc) > _GLOBAL_BEST_AUC[0]:
                            _GLOBAL_BEST_AUC[0] = float(best_val_auc)
                            torch.save(_pkg, f"optuna_models/lr_{TEST_PREF}_best.pkl")
                        wait = 0
                    else:
                        wait += 1
                
                    print(f'n = {n} | epoch {epoch+1:03d} | valid_auc {float(valid_auc):.4f} | test_auc {float(test_auc):.4f} | best {best_val_auc:.4f}| wait {wait} ')
                
                    # 早期終了（Optunaへ安全に通知）
                    if trial.should_prune() or epoch >= patience:
                        final_df = pd.read_csv(CSV_PATH, index_col = 0)
                        tmp_df = pd.DataFrame([{
                            "pref":n, "n":len(final_df), "epoch":best_epoch,"learning_rate":lr,"weight_decay":weight_decay,"batch_size":batch_size,
                            "m1_size":m1_size,"m2_size":m2_size, "dropout":dropout,"base_channels":base_ch,
                            "train_y":0, "train_acc":train_acc, "train_positive":train_positive, "train_meanscore":train_meanscore, "train_prauc":best_train_prauc,
                            "valid_y":0,"valid_acc":best_val_acc,"valid_auc":best_val_auc,"CNN_valid_auc":best_val_auc, "valid_positive":valid_positive, "valid_meanscore":valid_meanscore, "valid_prauc":best_val_prauc,
                            "test_y":0, "test_acc":best_test_acc, "test_auc":best_test_auc,"CNN_test_auc":best_test_auc,"test_positive":test_positive,  "test_meanscore":test_meanscore, "test_prauc":best_test_prauc
                        }])
                    
                        final_df = pd.concat([final_df, tmp_df], ignore_index=True)
                        final_df.to_csv(CSV_PATH)
                        datalist = [train_loss_list, valid_loss_list, test_loss_list,
                                    train_acc_list, valid_acc_list,test_acc_list,
                                    train_positive_list, valid_positive_list, test_positive_list,
                                    train_F1_list, valid_F1_list, test_F1_list,
                                    train_auc_list, valid_auc_list, test_auc_list,
                                    train_meanscore_list, valid_meanscore_list, test_meanscore_list, train_prauc_list, valid_prauc_list, test_prauc_list]
                        seve_metrics(f"{TEST_PREF}-lr-optina-植生あり" + str(len(final_df)-1) , datalist = datalist)
                        # prune を投げる → attempt ループの外にそのまま伝搬（OOM ではないのでリトライしない）
                        raise optuna.TrialPruned()

                # 正常終了時
                del model
                del optimizer
                torch.cuda.empty_cache()
                return best_val_auc

            # ===== OOM を捕捉してリトライ =====
            except torch.cuda.OutOfMemoryError as e:
                print(f"[Trial {trial.number}] CUDA OOM on attempt {attempt}/{MAX_RETRIES_PER_TRIAL}: {e}")
            except RuntimeError as e:
                if "out of memory" in str(e).lower():
                    print(f"[Trial {trial.number}] RuntimeError OOM on attempt {attempt}/{MAX_RETRIES_PER_TRIAL}: {e}")
                else:
                    # 別の RuntimeError はそのまま投げる
                    raise

            # ここに来た時点で OOM とみなす
            torch.cuda.empty_cache()
            gc.collect()

            if attempt == MAX_RETRIES_PER_TRIAL:
                print(f"[Trial {trial.number}] Reached max OOM retries. Pruning trial.")
                raise optuna.TrialPruned()

            print(f"[Trial {trial.number}] Waiting {WAIT_SECONDS} seconds before retry...")
            time.sleep(WAIT_SECONDS)


    # ---- このfold用のStudyを作成・実行 ----
    study = optuna.create_study(
        study_name=f"fold_{n}",
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=SEED),
        pruner=MedianPruner(n_startup_trials=5, n_warmup_steps=5),
    )
    study.optimize(objective, n_trials=5, gc_after_trial=True, n_jobs=1)



[I 2026-06-08 14:29:21,800] A new study created in memory with name: fold_0


Training on 0 th set
[Trial 0] Attempt 1/5
n = 0 | epoch 001 | valid_auc 0.6798 | test_auc 0.5776 | best 0.6798| wait 0 
n = 0 | epoch 002 | valid_auc 0.6366 | test_auc 0.5779 | best 0.6798| wait 1 
n = 0 | epoch 003 | valid_auc 0.6635 | test_auc 0.5961 | best 0.6798| wait 2 
n = 0 | epoch 004 | valid_auc 0.6682 | test_auc 0.5989 | best 0.6798| wait 3 
n = 0 | epoch 005 | valid_auc 0.6364 | test_auc 0.5869 | best 0.6798| wait 4 
n = 0 | epoch 006 | valid_auc 0.6224 | test_auc 0.5944 | best 0.6798| wait 5 
n = 0 | epoch 007 | valid_auc 0.6548 | test_auc 0.5842 | best 0.6798| wait 6 
n = 0 | epoch 008 | valid_auc 0.6240 | test_auc 0.5825 | best 0.6798| wait 7 
n = 0 | epoch 009 | valid_auc 0.5824 | test_auc 0.5579 | best 0.6798| wait 8 
n = 0 | epoch 010 | valid_auc 0.6235 | test_auc 0.5823 | best 0.6798| wait 9 


[I 2026-06-08 15:19:00,261] Trial 0 pruned. 


n = 0 | epoch 011 | valid_auc 0.6436 | test_auc 0.5969 | best 0.6798| wait 10 
[Trial 1] Attempt 1/5
n = 0 | epoch 001 | valid_auc 0.6951 | test_auc 0.6027 | best 0.6951| wait 0 
n = 0 | epoch 002 | valid_auc 0.6839 | test_auc 0.6005 | best 0.6951| wait 1 
n = 0 | epoch 003 | valid_auc 0.6794 | test_auc 0.6044 | best 0.6951| wait 2 
n = 0 | epoch 004 | valid_auc 0.6714 | test_auc 0.5993 | best 0.6951| wait 3 
n = 0 | epoch 005 | valid_auc 0.6277 | test_auc 0.5979 | best 0.6951| wait 4 
n = 0 | epoch 006 | valid_auc 0.6738 | test_auc 0.6023 | best 0.6951| wait 5 
n = 0 | epoch 007 | valid_auc 0.6711 | test_auc 0.5960 | best 0.6951| wait 6 
n = 0 | epoch 008 | valid_auc 0.6803 | test_auc 0.5928 | best 0.6951| wait 7 
n = 0 | epoch 009 | valid_auc 0.6755 | test_auc 0.5957 | best 0.6951| wait 8 
n = 0 | epoch 010 | valid_auc 0.6704 | test_auc 0.5941 | best 0.6951| wait 9 
n = 0 | epoch 011 | valid_auc 0.6607 | test_auc 0.5870 | best 0.6951| wait 10 


[I 2026-06-08 16:03:38,144] Trial 1 pruned. 


...

n = 0 | epoch 011 | valid_auc 0.5036 | test_auc 0.5000 | best 0.5115| wait 6 
[Trial 3] Attempt 1/5
n = 0 | epoch 001 | valid_auc 0.4966 | test_auc 0.5000 | best 0.4966| wait 0 
n = 0 | epoch 002 | valid_auc 0.5056 | test_auc 0.5000 | best 0.5056| wait 0 
n = 0 | epoch 003 | valid_auc 0.5027 | test_auc 0.5000 | best 0.5056| wait 1 
n = 0 | epoch 004 | valid_auc 0.5059 | test_auc 0.5000 | best 0.5059| wait 0 
n = 0 | epoch 005 | valid_auc 0.5091 | test_auc 0.5000 | best 0.5091| wait 0 
n = 0 | epoch 006 | valid_auc 0.5090 | test_auc 0.5000 | best 0.5091| wait 1 
n = 0 | epoch 007 | valid_auc 0.5107 | test_auc 0.5000 | best 0.5107| wait 0 
n = 0 | epoch 008 | valid_auc 0.5088 | test_auc 0.5000 | best 0.5107| wait 1 
n = 0 | epoch 009 | valid_auc 0.5127 | test_auc 0.5000 | best 0.5127| wait 0 
n = 0 | epoch 010 | valid_auc 0.5130 | test_auc 0.5000 | best 0.5130| wait 0 


[I 2026-06-08 17:49:19,031] Trial 3 pruned. 


n = 0 | epoch 011 | valid_auc 0.5052 | test_auc 0.5000 | best 0.5130| wait 1 
[Trial 4] Attempt 1/5
n = 0 | epoch 001 | valid_auc 0.6561 | test_auc 0.6089 | best 0.6561| wait 0 
n = 0 | epoch 002 | valid_auc 0.6478 | test_auc 0.6087 | best 0.6561| wait 1 
n = 0 | epoch 003 | valid_auc 0.6751 | test_auc 0.5891 | best 0.6751| wait 0 
n = 0 | epoch 004 | valid_auc 0.6492 | test_auc 0.5869 | best 0.6751| wait 1 
n = 0 | epoch 005 | valid_auc 0.6779 | test_auc 0.5975 | best 0.6779| wait 0 
n = 0 | epoch 006 | valid_auc 0.6515 | test_auc 0.5856 | best 0.6779| wait 1 
n = 0 | epoch 007 | valid_auc 0.6586 | test_auc 0.5860 | best 0.6779| wait 2 
n = 0 | epoch 008 | valid_auc 0.6694 | test_auc 0.5905 | best 0.6779| wait 3 
n = 0 | epoch 009 | valid_auc 0.6844 | test_auc 0.5949 | best 0.6844| wait 0 
n = 0 | epoch 010 | valid_auc 0.6629 | test_auc 0.5964 | best 0.6844| wait 1 


[I 2026-06-08 18:35:01,811] Trial 4 pruned. 


n = 0 | epoch 011 | valid_auc 0.6398 | test_auc 0.5725 | best 0.6844| wait 2 
